# Gram Schmidt



In [6]:
import time
import sys
from pathlib import Path
import numpy as np
from scipy.sparse import csr_matrix, issparse

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start the notebook inside Numerical_Linear_Algebra_Algorithms")
sys.path.insert(0, str(ROOT))
from common import loader, data_path, lu_solve, relative_residual
from Orthogonalization.gram_schmidt import gram_schmidt


def timed(fn, *args):
    start = time.perf_counter()
    result = fn(*args)
    return result, time.perf_counter() - start


## 3×3


In [7]:
A = np.array([[4., 1., 0.], [1., 3., 1.], [0., 1., 2.]])
x_true = np.ones(3)
b = A @ x_true
rng = np.random.default_rng(42)
Z = rng.normal(size=(A.shape[1], 3))
_ = gram_schmidt(A)
result, seconds = timed(gram_schmidt, A)
print("Seconds:", seconds)
Q, R = result
print("Factorization probe error:", float(np.linalg.norm(A @ Z - Q @ (R @ Z)) / np.linalg.norm(A @ Z)))
print("Orthogonality probe error:", float(np.linalg.norm(Q.T @ (Q @ Z) - Z) / np.linalg.norm(Z)))
assert float(np.linalg.norm(Q.T @ (Q @ Z) - Z) / np.linalg.norm(Z)) < 1e-12


Seconds: 1.1832977179437876e-05
Factorization probe error: 9.014365776215566e-17
Orthogonality probe error: 6.820595562521237e-17


## SuiteSparse

Change MATRIX to an existing file. bcsstk09 is 1,083×1,083; bcsstk17 is 10,974×10,974; west1505 is nonsymmetric and requires pivoting. No 10^5 matrix exists in this repo. Dense factorization costs O(n³); run the optional 10^4 case deliberately.


In [8]:
MATRIX = "bcsstk09"
MAT_PATH = None
A = loader("suitesparse", path=data_path(MATRIX, MAT_PATH), dense=True)
print("Matrix:", MATRIX, "shape:", A.shape)
rng = np.random.default_rng(42)
Z = rng.normal(size=(A.shape[1], 3))


Matrix: bcsstk09 shape: (1083, 1083)


In [9]:
result, seconds = timed(gram_schmidt, A)
print("Seconds:", seconds)
Q, R = result
print("Factorization probe error:", float(np.linalg.norm(A @ Z - Q @ (R @ Z)) / np.linalg.norm(A @ Z)))
print("Orthogonality probe error:", float(np.linalg.norm(Q.T @ (Q @ Z) - Z) / np.linalg.norm(Z)))


Seconds: 0.8117363330093212
Factorization probe error: 6.148626741950652e-16
Orthogonality probe error: 5.150833084606059e-06
